In [1]:
%reload_ext autoreload
%autoreload 2 --print

In [2]:
import pandas as pd
from src.paths import TRANSFORMED_DATA_DIR

df= pd.read_parquet(TRANSFORMED_DATA_DIR/"tabular_data.parquet")
df

,rides_previous_672_hour,rides_previous_671_hour,rides_previous_670_hour,rides_previous_669_hour,rides_previous_668_hour,rides_previous_667_hour,rides_previous_666_hour,rides_previous_665_hour,rides_previous_664_hour,rides_previous_663_hour,...,rides_previous_7_hour,rides_previous_6_hour,rides_previous_5_hour,rides_previous_4_hour,rides_previous_3_hour,rides_previous_2_hour,rides_previous_1_hour,pickup_hour,location_id,target_rides_next_hour
0,2.0,3.0,2.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,2026-01-29,3,0.0
1,0.0,0.0,0.0,0.0,1.0,0.0,2.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2026-01-30,3,0.0
2,0.0,0.0,0.0,0.0,2.0,0.0,0.0,1.0,0.0,0.0,...,0.0,2.0,2.0,0.0,0.0,1.0,0.0,2026-01-31,3,0.0
3,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,2.0,2026-02-01,3,0.0
4,0.0,0.0,0.0,0.0,1.0,1.0,2.0,2.0,1.0,2.0,...,0.0,3.0,2.0,1.0,0.0,0.0,0.0,2026-02-02,3,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64185,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-09-26,199,0.0
64186,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-09-27,199,0.0
64187,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-09-28,199,0.0
64188,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-09-29,199,0.0


In [3]:
from datetime import datetime
from src.data_split import train_test_split

X_train, y_train, X_test, y_test = train_test_split(
    df,
    cutoff_date= datetime(2026, 7, 1, 0, 0),
    target_column_name= "target_rides_next_hour"

)

print(f"{X_train.shape=}")
print(f"{X_test.shape=}")
print(f"{y_train.shape=}")
print(f"{y_test.shape=}")

X_train.shape=(40086, 674)
X_test.shape=(24104, 674)
y_train.shape=(40086,)
y_test.shape=(24104,)


In [4]:
# adding the first additional feature: rides averaged over past weeks

def average_rides_last_4_weeks(X: pd.DataFrame) -> pd.DataFrame:
    """
    adds one new column with average rides over 7, 14, 21, and 28 days
    """
    X["average_rides_last_4_weeks"]= 0.25 * (
                X[f"rides_previous_{7*24}_hour"] +\
                X[f"rides_previous_{2*7*24}_hour"] +\
                X[f"rides_previous_{3*7*24}_hour"] +\
                X[f"rides_previous_{4*7*24}_hour"]
    )

    return X

In [5]:
# wrap the function to turn it into a scikit-learn data tranformation step
from sklearn.preprocessing import FunctionTransformer
add_average_rides_last_4_weeks= FunctionTransformer(
    average_rides_last_4_weeks, validate= False)


In [6]:
add_average_rides_last_4_weeks.fit_transform(X_train)

,rides_previous_672_hour,rides_previous_671_hour,rides_previous_670_hour,rides_previous_669_hour,rides_previous_668_hour,rides_previous_667_hour,rides_previous_666_hour,rides_previous_665_hour,rides_previous_664_hour,rides_previous_663_hour,...,rides_previous_7_hour,rides_previous_6_hour,rides_previous_5_hour,rides_previous_4_hour,rides_previous_3_hour,rides_previous_2_hour,rides_previous_1_hour,pickup_hour,location_id,average_rides_last_4_weeks
0,2.0,3.0,2.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,2026-01-29,3,0.75
1,0.0,0.0,0.0,0.0,1.0,0.0,2.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2026-01-30,3,0.00
2,0.0,0.0,0.0,0.0,2.0,0.0,0.0,1.0,0.0,0.0,...,0.0,2.0,2.0,0.0,0.0,1.0,0.0,2026-01-31,3,0.00
3,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,2.0,2026-02-01,3,0.00
4,0.0,0.0,0.0,0.0,1.0,1.0,2.0,2.0,1.0,2.0,...,0.0,3.0,2.0,1.0,0.0,0.0,0.0,2026-02-02,3,0.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40081,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-06-26,199,0.00
40082,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-06-27,199,0.00
40083,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-06-28,199,0.00
40084,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-06-29,199,0.00


In [ ]:
# adding the second additional feature: the day of the week and the hour of the day from the pickup date
# we do this by the most customisable way to define data transformations in scikit-learn: by creating a class that inherits from other classes in scikit-learn
from sklearn.base import BaseEstimator, TransformerMixin

class TemporalFeatureEngineer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        X_= X.copy()

        # Generate numeric columns from datetime
        X_["hour"]= X_["pickup_hour"].dt.hour
        X_["day_of_week"]= X_["pickup_hour"].dt.dayofweek

        return X_.drop(columns=["pickup_hour"])
    


In [8]:
add_temporal_features= TemporalFeatureEngineer()
add_temporal_features.fit_transform(X_train)

,rides_previous_672_hour,rides_previous_671_hour,rides_previous_670_hour,rides_previous_669_hour,rides_previous_668_hour,rides_previous_667_hour,rides_previous_666_hour,rides_previous_665_hour,rides_previous_664_hour,rides_previous_663_hour,...,rides_previous_6_hour,rides_previous_5_hour,rides_previous_4_hour,rides_previous_3_hour,rides_previous_2_hour,rides_previous_1_hour,location_id,average_rides_last_4_weeks,hour,day_of_week
0,2.0,3.0,2.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,3,0.75,0,3
1,0.0,0.0,0.0,0.0,1.0,0.0,2.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,3,0.00,0,4
2,0.0,0.0,0.0,0.0,2.0,0.0,0.0,1.0,0.0,0.0,...,2.0,2.0,0.0,0.0,1.0,0.0,3,0.00,0,5
3,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,2.0,3,0.00,0,6
4,0.0,0.0,0.0,0.0,1.0,1.0,2.0,2.0,1.0,2.0,...,3.0,2.0,1.0,0.0,0.0,0.0,3,0.50,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40081,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,199,0.00,0,4
40082,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,199,0.00,0,5
40083,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,199,0.00,0,6
40084,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,199,0.00,0,0


In [ ]:
# third additional feature: encoding the pickup location as a categorical variable using latitude and longitude



In [9]:
import lightgbm as lgb

In [11]:
# combining the two feature engineering steps and the LightGBM model into a single pipeline

from sklearn.pipeline import make_pipeline

pipeline= make_pipeline(
    add_average_rides_last_4_weeks,
    add_temporal_features,
    lgb.LGBMRegressor()
)

pipeline.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.650001 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 160268
[LightGBM] [Info] Number of data points in the train set: 40086, number of used features: 675
[LightGBM] [Info] Start training from score 15.125331


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('functiontransformer', ...), ('temporalfeatureengineer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](675,)","['rides_previous_672_hour','rides_previous_671_hour', 'rides_previous_670_hour',...,'pickup_hour','location_id', 'average_rides_last_4_weeks']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,675
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function ave...00208DF490A40>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False


In [12]:
predictions= pipeline.predict(X_test)
predictions

array([0.23492586, 0.28092754, 0.23492586, ..., 0.23492586, 0.23492586,
       0.23492586], shape=(24104,))

In [13]:
from sklearn.metrics import mean_absolute_error
test_mae= mean_absolute_error(y_test, predictions)
print(f"{test_mae= :.4f}")

test_mae= 3.6593
